# Seeded LSB Audio Steganography
Recovered implementation. Saved outputs and account metadata were removed for publication. Input data is not bundled.


In [ ]:
!pip install numpy scipy

In [ ]:
import os

# Create the folder if it doesn't exist
os.makedirs('./Ex3_files', exist_ok=True)

from google.colab import files
import os

uploaded = files.upload()  # Choose your files from your computer

# Move uploaded files to the correct folder
os.makedirs('./Ex3_files', exist_ok=True)
for filename in uploaded.keys():
    os.rename(filename, f'./Ex3_files/{filename}')


In [ ]:
import numpy as np
import scipy.io.wavfile as wav
import random

In [ ]:
folder_path = './Ex3_files/'
input_audio = folder_path + 'Ex3_sound5.wav'
output_audio = folder_path + 'stego_audio.wav'

In [ ]:
message = "An eye for an eye makes the whole world blind"
# Add a delimiter to indicate message end
message += "###"

# Convert message to binary string
binary_message = ''.join(format(ord(char), '08b') for char in message)
print(f"Total bits to embed: {len(binary_message)}")


In [ ]:
# Read audio
sr, data = wav.read(input_audio)

# Ensure mono for simplicity
if data.ndim > 1:
    data = data[:, 0]

# Flatten to int32 for safe bitwise operations
audio = data.astype(np.int32)

# Generate random, non-repeating indices for embedding
random.seed(42)  # For reproducibility
indices = random.sample(range(len(audio)), len(binary_message))

# Embed bits
for i, bit in enumerate(binary_message):
    audio[indices[i]] = (audio[indices[i]] & ~1) | int(bit)

# Convert back to original format
stego_audio = np.clip(audio, -32768, 32767).astype(np.int16)

# Save stego audio
wav.write(output_audio, sr, stego_audio)
print("✅ Message embedded and audio saved.")


In [ ]:
def extract_message(audio_path, message_length):
    sr, data = wav.read(audio_path)

    if data.ndim > 1:
        data = data[:, 0]

    random.seed(42)
    indices = random.sample(range(len(data)), message_length)

    bits = [str(data[idx] & 1) for idx in indices]

    chars = [chr(int(''.join(bits[i:i+8]), 2)) for i in range(0, len(bits), 8)]
    return ''.join(chars)


In [ ]:
total_bits = len(binary_message)
extracted = extract_message(output_audio, total_bits)
print(f"Recovered message: {extracted}")
